# Hybrid Robotic Vision Pilot Experiments in Google Colab

This notebook sets up the **first stage** of the experimental pipeline for the hybrid visual telemetry paper.

It focuses on the **video-only baseline** first, which is the correct starting point before adding:
- ROI extraction
- JPEG XL still compression
- classification refinement
- hybrid bitrate accounting

## What this notebook does

1. Mounts Google Drive  
2. Creates the project folder structure  
3. Installs required tools (`ffmpeg`, `ffprobe`)  
4. Verifies the UAVDT dataset location  
5. Runs the **video-only encoding baseline** at:
   - **0.8 Mbps** (low bitrate regime)
   - **1.4 Mbps** (moderate bitrate regime)
6. Inspects generated metrics and previews encoded outputs  
7. Optionally runs the baseline on multiple clips

## Expected project structure in Google Drive

```text
MyDrive/
  hybrid_robotic_vision/
    data/
      uavdt/
        videos/
        annotations/
    runs/
    notebooks/
    tools/
```

## Assumptions

- You already uploaded or copied the experiment scripts into:

```text
MyDrive/hybrid_robotic_vision/tools/exp_setup/
```

- The script `01_encode_video_only.py` exists there.
- UAVDT video files are placed in:

```text
MyDrive/hybrid_robotic_vision/data/uavdt/videos/
```

## Notes

- Google Colab runtimes are temporary, so all important outputs should be written to **Google Drive**.
- For the **video-only step**, direct writing to Drive is fine.
- Later, when extracting many ROI crops, it may be better to process files in `/content` and copy results back to Drive.

## 1. Mount Google Drive

Run this first so the notebook can access your persistent project files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Create the folder structure

This cell creates the full project structure in Drive if it does not already exist.
It is safe to run multiple times.

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/hybrid_robotic_vision")

folders = [
    ROOT / "data" / "uavdt" / "videos",
    ROOT / "data" / "uavdt" / "annotations",
    ROOT / "runs",
    ROOT / "notebooks",
    ROOT / "tools",
]

for p in folders:
    p.mkdir(parents=True, exist_ok=True)

print("Created / verified folder structure under:")
print(ROOT)

print("\nFolders:")
for p in folders:
    print("-", p)

## 3. Define project paths

These variables are reused throughout the notebook.

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/hybrid_robotic_vision")
DATA = ROOT / "data" / "uavdt"
VIDEOS = DATA / "videos"
ANNOTATIONS = DATA / "annotations"
RUNS = ROOT / "runs"
TOOLS = ROOT / "tools" / "exp_setup"

print("ROOT:", ROOT)
print("VIDEOS:", VIDEOS)
print("ANNOTATIONS:", ANNOTATIONS)
print("RUNS:", RUNS)
print("TOOLS:", TOOLS)

## 4. Install required system tools

For the first stage of the pipeline, we only need:
- `ffmpeg`
- `ffprobe`

These are used by the video-only encoder script.

In [ ]:
!apt-get update -qq
!apt-get install -y ffmpeg

In [ ]:
!ffmpeg -version | head -n 5
print()
!ffprobe -version | head -n 5

## 5. Verify the experiment code exists

This checks that your scaffolded code directory is available in Drive.

Expected key file:
- `01_encode_video_only.py`

In [ ]:
if not TOOLS.exists():
    raise FileNotFoundError(f"Tools folder not found: {TOOLS}")

print("Tools folder exists.")
print("\nContents:")
for p in sorted(TOOLS.iterdir()):
    print("-", p.name)

## 6. Move into the experiment code directory

This makes it easier to run scripts with relative imports.

In [ ]:
%cd /content/drive/MyDrive/hybrid_robotic_vision/tools/exp_setup

## 7. Inspect available UAVDT videos

This lists the files currently available in the UAVDT video folder.

If nothing appears here, upload or copy a few UAVDT clips into:

```text
MyDrive/hybrid_robotic_vision/data/uavdt/videos/
```

In [ ]:
video_files = sorted([p for p in VIDEOS.iterdir() if p.is_file()])

print(f"Found {len(video_files)} file(s) in {VIDEOS}")
for p in video_files[:50]:
    print("-", p.name)

## 8. Select a clip for the first smoke test

Edit `TEST_CLIP_NAME` below if needed.

The notebook will:
- verify that the file exists,
- then use it for the low and moderate bitrate runs.

In [ ]:
TEST_CLIP_NAME = None  # Example: "M0101.mp4"

if TEST_CLIP_NAME is None:
    if len(video_files) == 0:
        raise RuntimeError("No video files found in the UAVDT videos folder.")
    TEST_CLIP = video_files[0]
    print("TEST_CLIP_NAME was not set manually.")
    print("Using the first detected file:", TEST_CLIP.name)
else:
    TEST_CLIP = VIDEOS / TEST_CLIP_NAME
    if not TEST_CLIP.exists():
        raise FileNotFoundError(f"Selected test clip not found: {TEST_CLIP}")

print("\nResolved test clip:")
print(TEST_CLIP)

## 9. Optional: inspect the raw input clip

This is useful to verify:
- the file opens correctly,
- the clip looks like a valid UAVDT sequence,
- the original resolution / fps is reasonable.

In [ ]:
from IPython.display import Video

Video(str(TEST_CLIP), embed=True)

## 10. Run the video-only baseline at **0.8 Mbps**

This corresponds to the **low bitrate regime** in the paper design.

Expected output location:

```text
runs/uavdt_low/video_only/<clip_stem>/
```

In [ ]:
LOW_OUTPUT_DIR = RUNS / "uavdt_low" / "video_only"

!python 01_encode_video_only.py \
  --input "{TEST_CLIP}" \
  --output-dir "{LOW_OUTPUT_DIR}" \
  --bitrate-mbps 0.8

## 11. Run the video-only baseline at **1.4 Mbps**

This corresponds to the **moderate bitrate regime** in the paper design.

In [ ]:
MODERATE_OUTPUT_DIR = RUNS / "uavdt_moderate" / "video_only"

!python 01_encode_video_only.py \
  --input "{TEST_CLIP}" \
  --output-dir "{MODERATE_OUTPUT_DIR}" \
  --bitrate-mbps 1.4

## 12. Inspect generated outputs

This shows where the script wrote files for the current test clip.

In [ ]:
clip_stem = TEST_CLIP.stem

low_clip_dir = LOW_OUTPUT_DIR / clip_stem
moderate_clip_dir = MODERATE_OUTPUT_DIR / clip_stem

print("Low bitrate output dir:", low_clip_dir)
print("Moderate bitrate output dir:", moderate_clip_dir)

print("\nLow bitrate contents:")
if low_clip_dir.exists():
    for p in sorted(low_clip_dir.iterdir()):
        print("-", p.name)
else:
    print("Directory not found.")

print("\nModerate bitrate contents:")
if moderate_clip_dir.exists():
    for p in sorted(moderate_clip_dir.iterdir()):
        print("-", p.name)
else:
    print("Directory not found.")

## 13. Load and inspect the metrics JSON files

This is the main sanity check for the first stage.

You want to confirm:
- the achieved bitrate is near the target,
- output resolution is **1280×720**,
- fps is **15**,
- the codec is correct.

In [ ]:
import json

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

low_metrics_path = low_clip_dir / "video_only_metrics.json"
moderate_metrics_path = moderate_clip_dir / "video_only_metrics.json"

low_metrics = load_json(low_metrics_path)
moderate_metrics = load_json(moderate_metrics_path)

print("LOW BITRATE METRICS")
print(json.dumps(low_metrics, indent=2))

print("\nMODERATE BITRATE METRICS")
print(json.dumps(moderate_metrics, indent=2))

## 14. Preview encoded outputs

This helps visually confirm that:
- the videos were encoded correctly,
- the low-bitrate result is visibly more compressed,
- both files are playable.

In [ ]:
from IPython.display import Video, display

low_video_candidates = [p for p in low_clip_dir.iterdir() if p.suffix.lower() in [".mp4", ".mkv", ".mov"]]
moderate_video_candidates = [p for p in moderate_clip_dir.iterdir() if p.suffix.lower() in [".mp4", ".mkv", ".mov"]]

print("Low bitrate encoded file:", low_video_candidates[0] if low_video_candidates else "Not found")
print("Moderate bitrate encoded file:", moderate_video_candidates[0] if moderate_video_candidates else "Not found")

In [ ]:
if low_video_candidates:
    print("Low bitrate preview")
    display(Video(str(low_video_candidates[0]), embed=True))

if moderate_video_candidates:
    print("Moderate bitrate preview")
    display(Video(str(moderate_video_candidates[0]), embed=True))

## 15. Create a compact summary table

This makes it easier to compare the two regimes side by side.

In [ ]:
import pandas as pd

summary_rows = [
    {
        "regime": "low",
        "target_bitrate_mbps": low_metrics.get("target_bitrate_mbps"),
        "achieved_bitrate_mbps": low_metrics.get("achieved_bitrate_mbps"),
        "duration_s": low_metrics.get("duration_s"),
        "size_bytes": low_metrics.get("size_bytes"),
        "fps": low_metrics.get("fps"),
        "width": low_metrics.get("width"),
        "height": low_metrics.get("height"),
        "codec": low_metrics.get("codec"),
    },
    {
        "regime": "moderate",
        "target_bitrate_mbps": moderate_metrics.get("target_bitrate_mbps"),
        "achieved_bitrate_mbps": moderate_metrics.get("achieved_bitrate_mbps"),
        "duration_s": moderate_metrics.get("duration_s"),
        "size_bytes": moderate_metrics.get("size_bytes"),
        "fps": moderate_metrics.get("fps"),
        "width": moderate_metrics.get("width"),
        "height": moderate_metrics.get("height"),
        "codec": moderate_metrics.get("codec"),
    },
]

summary_df = pd.DataFrame(summary_rows)
summary_df

## 16. Batch run the video-only baseline on several clips

Once the smoke test succeeds, use this section to process a small pilot subset.

Recommended first pilot:
- 3 clips for quick validation
- then 8–12 clips for the real pilot subset

Edit `CLIP_NAMES` below.

In [ ]:
CLIP_NAMES = [
    # Examples:
    # "M0101.mp4",
    # "M0203.mp4",
    # "M0401.mp4",
]

if len(CLIP_NAMES) == 0:
    print("No batch clips specified yet. Edit CLIP_NAMES to run multiple clips.")
else:
    print("Batch clip list:")
    for name in CLIP_NAMES:
        print("-", name)

In [ ]:
# Run the low-bitrate baseline on a batch of clips

for clip_name in CLIP_NAMES:
    clip_path = VIDEOS / clip_name
    if not clip_path.exists():
        print(f"Skipping missing file: {clip_path}")
        continue

    print(f"\nRunning LOW bitrate baseline for {clip_name}")
    !python 01_encode_video_only.py \
      --input "{clip_path}" \
      --output-dir "{LOW_OUTPUT_DIR}" \
      --bitrate-mbps 0.8

In [ ]:
# Run the moderate-bitrate baseline on a batch of clips

for clip_name in CLIP_NAMES:
    clip_path = VIDEOS / clip_name
    if not clip_path.exists():
        print(f"Skipping missing file: {clip_path}")
        continue

    print(f"\nRunning MODERATE bitrate baseline for {clip_name}")
    !python 01_encode_video_only.py \
      --input "{clip_path}" \
      --output-dir "{MODERATE_OUTPUT_DIR}" \
      --bitrate-mbps 1.4

## 17. Aggregate metrics across multiple clips

This section scans the output folders and builds a simple table for later analysis.

In [ ]:
import json
import pandas as pd

def collect_metrics(run_dir, regime_name):
    rows = []
    if not run_dir.exists():
        return rows

    for clip_dir in sorted(run_dir.iterdir()):
        if not clip_dir.is_dir():
            continue
        metrics_path = clip_dir / "video_only_metrics.json"
        if metrics_path.exists():
            with open(metrics_path, "r") as f:
                d = json.load(f)
            d["clip"] = clip_dir.name
            d["regime"] = regime_name
            rows.append(d)
    return rows

all_rows = []
all_rows += collect_metrics(LOW_OUTPUT_DIR, "low")
all_rows += collect_metrics(MODERATE_OUTPUT_DIR, "moderate")

metrics_df = pd.DataFrame(all_rows)
metrics_df

## 18. Save the aggregated metrics table

This writes a CSV to Drive for later analysis.

In [ ]:
if len(metrics_df) > 0:
    summary_csv = RUNS / "video_only_metrics_summary.csv"
    metrics_df.to_csv(summary_csv, index=False)
    print("Saved metrics summary to:")
    print(summary_csv)
else:
    print("No metrics collected yet.")

## 19. What to do next

Once the video-only baseline works, the next notebook should implement:

1. **Detection/tracking export**
   - run detector on decoded base videos
   - save track CSVs

2. **ROI extraction**
   - crop original-frame object boxes
   - attach metadata

3. **JPEG XL still compression**
   - encode selected ROI crops
   - measure still-image bit cost

4. **Classification collection**
   - classify decoded-video crops
   - classify JPEG XL ROI crops
   - compare object-level accuracy at equal total bitrate

## First milestone checklist

Before moving on, confirm:
- [ ] one clip successfully encoded at 0.8 Mbps
- [ ] one clip successfully encoded at 1.4 Mbps
- [ ] metrics JSON files look correct
- [ ] output videos are playable
- [ ] batch run works on at least 3 clips